# 到了切块。之前我给过一个经验起点：中文场景 chunk_size=500、overlap=50。那个数字能让你"跑起来"。今天讲的是，它在真实文档上为什么会翻车，以及怎么救。

# 固定长度切块，有三宗罪：

# 第一宗，切飞句子。500 字硬切，刀口正好落在一句话中间。前半句留在块 A，后半句去了块 B，两边都是残废的半句。

# 第二宗，切散表格和代码块。一张十行的表格，被从第五行拦腰斩断，变成两个"半张表"。召回到哪一半都答不全——这正是开头翻车现场的根因之一。

# 第三宗，切断上下文。标题"三、退换货政策"留在块 A，具体条款去了块 B。当你召回到 B，它是一段没头没尾的条款，模型不知道这属于哪一节、针对哪款产品。

# 对应三层补救，从轻到重。

In [1]:
text = """1、勇气：2023年9月，全网都警告妙瓦底KK园区极度危险、国人进入极难脱身，舆论普遍劝诫所有人远离此地，他依旧主动深入妙瓦底，经历惊心动魄的36小时险境；
2024‑10‑21俄乌冲突战火未熄之时，他奔赴乌克兰基辅、第聂伯罗等前线周边区域实地探访，在当地遭遇华人绑架团伙盯上、人身安全受到直接威胁，依旧坚持完成一线见闻记录。

2、求真：2023年从妙瓦底返回之后，网上大量流传“园区里受害者全部是被拐骗而来”的单一论调，他基于自己36小时的实地观察，
公开提出一部分人属于自愿前往的观点，没有为迎合大众情绪修改自己亲眼得到的见闻，坚持把现场事实完整放出，还主动提出可以提供实拍照片佐证自己的经历。

3、赤诚：在乌克兰第聂伯罗走访期间，
他专门前往难民孤儿之家看望战火下流离失所的孩童，
记录战争平民受害者真实生存状态；海外多地行走全程始终高举中国护照，对外清晰表明自身中国人身份，即便后来因为乌克兰入境记录，在俄罗斯机场遭到边检扣留24小时并被永久拒签，也不曾回避自己过往一线探访经历吧，对不对。"""



In [2]:



# 第一层，递归切块。RecursiveCharacterTextSplitter 不再一刀切，而是按分隔符层级优先在语义边界下手——先试着按段落切，太长再按句子，再不行才按词。它治的是第一宗罪。

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=30,
    separators=["\n\n", "\n", "。", "！", "？", "；", " ", ""],  # 中文友好的分隔符层级
)
chunks = splitter.split_text(text)

# 工作逻辑
# 1.先看文本长度，如果 ≤ chunk_size → **不切分，直接一块返回，结束！**
# 2. 拿最高优先级分隔符 `\n\n` 切割文本
# 3. 切出来的每一小块，如果 ≤ chunk_size (500) → 保留，结束
# 4. 如果某一块**超过 500 字符** → 递归，用**下一级分隔符 (`\n`)**继续切这块超长文本
# 5. 一直往下试，直到最后兜底 `""`，强制把超长文本剪开
# 注意：`\n` 是 1 个字符，会被计入 chunk_size 长度**。

# 打印结果
print(f"一共分出 {len(chunks)} 个块\n")
for i, chunk in enumerate(chunks):
    print(f"==== 块{i+1} ====")
    print(chunk)
    print()


d:\Anaconda\envs\llamaindexRAG\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


一共分出 8 个块

==== 块1 ====
1、勇气：2023年9月，全网都警告妙瓦底KK园区极度危险、国人进入极难脱身，舆论普遍劝诫所有人远离此地，他依旧主动深入妙瓦底，经历惊心动魄的36小时险境；

==== 块2 ====
2024‑10‑21俄乌冲突战火未熄之时，他奔赴乌克兰基辅、第聂伯罗等前线周边区域实地探访，在当地遭遇华人绑架团伙盯上、人身安全受到直接威胁，依旧坚持完成一线见闻记录。

==== 块3 ====
2、求真：2023年从妙瓦底返回之后，网上大量流传“园区里受害者全部是被拐骗而来”的单一论调，他基于自己36小时的实地观察，

==== 块4 ====
公开提出一部分人属于自愿前往的观点，没有为迎合大众情绪修改自己亲眼得到的见闻，坚持把现场事实完整放出，还主动提出可以提供实拍照片佐证自己的经历。

==== 块5 ====
3、赤诚：在乌克兰第聂伯罗走访期间，
他专门前往难民孤儿之家看望战火下流离失所的孩童，

==== 块6 ====
记录战争平民受害者真实生存状态

==== 块7 ====
；海外多地行走全程始终高举中国护照，对外清晰表明自身中国人身份，即便后来因为乌克兰入境记录，在俄罗斯机场遭到边检扣留24小时并被永久拒签，也不曾回避自己过往一线探访经历吧，对不对

==== 块8 ====
。



In [3]:
# ============ markdown_text 输入样例 ============
markdown_text = """# 人工智能介绍
人工智能是当下非常热门的技术方向。它深刻影响各行各业的发展。

## 机器学习
机器学习属于人工智能的一个重要分支，拥有广阔的应用前景。

### 监督学习
监督学习是使用标注好的数据来训练模型。

### 无监督学习
无监督学习不需要人工标注标签。

## 深度学习
深度学习是机器学习下的子领域，最近几年发展速度飞快。
"""

In [ ]:
# 第二层，按文档结构切。如果你在解析阶段保留了标题层级（还记得 unstructured 的 Title 类别、Markdown 的 # 吗），
# 就能用 MarkdownHeaderTextSplitter 按标题切，每一块自动带上"标题路径"作为 metadata。它治的是第三宗罪。

from langchain_text_splitters import MarkdownHeaderTextSplitter

headers = [("#", "h1"), ("##", "h2"), ("###", "h3")]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers)
docs = md_splitter.split_text(markdown_text)

# 每个 doc 自带标题路径，例如：
# doc.metadata == {"h1": "产品手册", "h2": "售后服务", "h3": "退换货政策"}
# 召回时把这条路径拼回正文，模型就知道这段话的归属

# 打印分割结果
for doc in docs:
    print("-"*50)
    print("元数据(标题信息):", doc.metadata)
    print("文本内容:\n", doc.page_content)

第三层，语义切块。用 embedding 去算相邻句子的相似度，在"话题发生切换"的地方下刀。它最贴语义，但也最慢、最贵——每句都要过一遍 embedding。
2026/8/23目前用的有问题，要么是向量模型不对，要么是这个SemanticChunker有问题，不建议用这种办法

In [74]:
text = """夏天冰镇西瓜吃起来清甜解渴。
放进冰箱冷藏两小时口感更佳。
Java是一门面向对象的编程语言。
后端服务器很多业务系统都是使用Java开发。aaaaaaaaaaaaaaaaaaabbbbbbbbbbbbbbbbbbbbbbbbbbnnnnnnnnnnnnnnnnnnnnnnmmmmmmmmmmmmmmmmmmm
青藏高原平均海拔4000米以上。
那里空气稀薄，昼夜温差非常大。
篮球比赛每支队伍上场五名球员。
投篮命中三分线外可以拿到三分。"""


In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
# 复用已有的 bge-m3 embedding
# 方式1：LangChain包装一层（推荐）
embeddings = HuggingFaceEmbeddings(
    model_name=r"G:\力扣代码集\langchain实现RAG\download_model\m3e-base",
)

chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="standard_deviation",
    breakpoint_threshold_amount=0.1,
)
chunks = chunker.split_text(text)

print(f"一共切出来 {len(chunks)} 块\n")
for idx, one_chunk in enumerate(chunks):
    print(f"====== 块 {idx+1} ======")
    print(one_chunk)
    print("\n")

但不是越高级越好。结构化文档——带清晰标题的手册、Markdown——用第二层的 header split 性价比最高，又快又准。只有那种一大段连绵不断的散文，才值得动用语义切块这把重武器。选型看文档，别无脑上最贵的。